step1

In [9]:
import os
import pandas as pd

print("=== ステップ1: 豊洲エリアの対象トリップ抽出（列名逆転対応版） ===")

# 1. データの読み込み
path_df = pd.read_csv("../data/routes/Toyosu-2018-2021/Toyosu-2021/walk/path.csv")
walk_paths = path_df[path_df["Mode"].eq(500)].copy()
walk_paths["TripID"] = pd.to_numeric(walk_paths["TripID"], errors="coerce")
walk_paths = walk_paths.dropna(
    subset=[
        "TripID",
        "Path_Origin_nodeID",
        "Path_Destination_nodeID",
        "Path_Length(m)",
    ]
)
walk_paths["TripID"] = walk_paths["TripID"].astype(int)

# 2. NodeIDの重複を排除してノード座標辞書を作成
nodes = pd.read_csv("../data/network/tokyo-metropolitan-area/walk_node.csv", low_memory=False)
nodes_unique = nodes.drop_duplicates(subset=["NodeID"])

# ★【ポイント】このデータはカラム名と中身が逆（Lat列に経度、Lon列に緯度）になっているため、
# 辞書を作る段階で正しく入れ替えておく
node_coords = {}
for _, r in nodes_unique.iterrows():
  nid = int(r["NodeID"])
  # 実データのカラム名に合わせて修正: Lat列に経度が、Lon列に緯度が入っている
  actual_lon = float(r["Lat"])
  actual_lat = float(r["Lon"])
  node_coords[nid] = {"Lon": actual_lon, "Lat": actual_lat}


# 3. 豊洲エリアの判定関数（正しく直した座標でバウンディングボックス判定）
def is_toyosu_trip(row):
  o_node = int(row["Path_Origin_nodeID"])
  d_node = int(row["Path_Destination_nodeID"])
  if o_node in node_coords and d_node in node_coords:
    o_lon = node_coords[o_node]["Lon"]
    o_lat = node_coords[o_node]["Lat"]
    d_lon = node_coords[d_node]["Lon"]
    d_lat = node_coords[d_node]["Lat"]

    # 豊洲の範囲： 経度 139.775〜139.815, 緯度 35.635〜35.670
    in_toyosu_o = (139.775 <= o_lon <= 139.815) and (35.635 <= o_lat <= 35.670)
    in_toyosu_d = (139.775 <= d_lon <= 139.815) and (35.635 <= d_lat <= 35.670)
    return in_toyosu_o or in_toyosu_d
  return False


# 4. 豊洲エリアのトリップのみに厳選して保存
toyosu_mask = walk_paths.apply(is_toyosu_trip, axis=1)
toyosu_walk = walk_paths[toyosu_mask].reset_index(drop=True)

output_dir = "../data/kitagawa_generated"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "step1_toyosu_walk_trips.csv")
toyosu_walk.to_csv(output_path, index=False)

print(f"✅ 抽出完了: 豊洲エリアの対象トリップ数 = {len(toyosu_walk):,} 件")
print(f"保存先: {output_path}")

=== ステップ1: 豊洲エリアの対象トリップ抽出（列名逆転対応版） ===
✅ 抽出完了: 豊洲エリアの対象トリップ数 = 8,163 件
保存先: ../data/kitagawa_generated/step1_toyosu_walk_trips.csv


step2

In [10]:
import networkx as nx

print("=== ステップ2: ネットワークグラフ G の構築 ===")

links = pd.read_csv(
    "../data/network/tokyo-metropolitan-area/walk_link.csv", low_memory=False
)
G = nx.DiGraph()

for _, row in links.iterrows():
  u = int(row["ONodeID"])
  v = int(row["DNodeID"])
  length_m = float(row["length"]) * 1000.0
  dir_flag = (
      int(row["direction restrictions"])
      if pd.notnull(row["direction restrictions"])
      else 1
  )

  attr = {
      "link_id": row["LinkID"],
      "length_m": length_m,
      "weight": length_m,
      "is_stairs": (
          1
          if (
              row["Pavement structure code"] == -96
              or row["Station layout code"] == 1
          )
          else 0
      ),
      "is_bridge": 1 if row["Pavement structure code"] == 3 else 0,
      "is_no_sidewalk": (
          1
          if (
              row["Pavement width code"] == 1
              or row["Pavement structure code"] == -97
          )
          else 0
      ),
      "is_narrow": 1 if row["Pavement width code"] == 2 else 0,
  }
  if dir_flag in [1, 2]:
    G.add_edge(u, v, **attr)
  if dir_flag in [1, 3]:
    G.add_edge(v, u, **attr)

print(
    f"✅ グラフ構築完了 (ノード数: {G.number_of_nodes():,}, エッジ数:"
    f" {G.number_of_edges():,})"
)

=== ステップ2: ネットワークグラフ G の構築 ===
✅ グラフ構築完了 (ノード数: 141,197, エッジ数: 387,574)


step3

In [11]:
from itertools import islice
import os
import networkx as nx
import numpy as np
import pandas as pd

print("=== ステップ3: 進捗ログ付きルート統合処理（実測 ＋ 代替経路探索） ===")

# 1. ステップ1で正しく抽出された豊洲対象トリップデータを読み込む
step1_file = "../data/kitagawa_generated/step1_toyosu_walk_trips.csv"
if not os.path.exists(step1_file):
  raise FileNotFoundError(
      f"'{step1_file}' が見つかりません。先にステップ1を実行してください。"
  )

toyosu_walk = pd.read_csv(step1_file)
print(f"-> 読み込んだ豊洲対象トリップ数: {len(toyosu_walk):,} 件")

# 実測リンクデータの読み込み
link_df = pd.read_csv(
    "../data/routes/Toyosu-2018-2021/Toyosu-2021/walk/link.csv"
)
link_df["TripID"] = pd.to_numeric(link_df["TripID"], errors="coerce")

# 2. 統合レコードの作成（50件ごとに進捗ログを出力）
integrated_records = []
success_count = 0
skip_count = 0
total_trips = len(toyosu_walk)

print("\n=== トリップごとのルート統合処理（メインループ）を開始 ===")
for i, path_row in toyosu_walk.iterrows():
  trip_id = int(path_row["TripID"])
  o_node = int(path_row["Path_Origin_nodeID"])
  d_node = int(path_row["Path_Destination_nodeID"])

  # --- ★ 進捗ログ出力（50件ごと、または最後の件数に達したとき） ---
  if (i + 1) % 50 == 0 or (i + 1) == total_trips:
    print(
        f"⏳ 進捗: {i + 1} / {total_trips} 件処理完了 (成功: {success_count},"
        f" スキップ: {skip_count})"
    )

  # ① 【AltID = 1】: link.csv から実測通過リンクを取得
  observed_links = link_df[link_df["TripID"] == trip_id].sort_values(
      "Link_Dep_Time"
  )
  if len(observed_links) == 0:
    skip_count += 1
    continue

  for seq, (_, link_row) in enumerate(observed_links.iterrows(), start=1):
    link_len = (
        float(link_row["Link_Length(m)"])
        if pd.notnull(link_row["Link_Length(m)"])
        else 0.0
    )
    integrated_records.append({
        "TripID": trip_id,
        "AltID": 1,
        "Chosen": 1,
        "Seq": seq,
        "LinkID": link_row["LinkID"],
        "length_m": link_len,
    })

  # ② 【AltID = 2, 3】: 代替経路の探索
  if o_node == d_node or not G.has_node(o_node) or not G.has_node(d_node):
    skip_count += 1
    continue

  try:
    k_paths = list(
        islice(
            nx.shortest_simple_paths(G, o_node, d_node, weight="weight"), 3
        )
    )

    alt_assigned = 0
    for path in k_paths:
      if alt_assigned >= 2:
        break
      alt_assigned += 1

      for seq, (u, v) in enumerate(zip(path[:-1], path[1:]), start=1):
        edge_data = G[u][v]
        integrated_records.append({
            "TripID": trip_id,
            "AltID": alt_assigned + 1,
            "Chosen": 0,
            "Seq": seq,
            "LinkID": edge_data["link_id"],
            "length_m": edge_data["length_m"],
        })

    if alt_assigned >= 2:
      success_count += 1
    else:
      skip_count += 1

  except (nx.NetworkXNoPath, nx.NodeNotFound):
    skip_count += 1
    continue

# 3. 保存
output_dir = "../data/kitagawa_generated"
os.makedirs(output_dir, exist_ok=True)
output_filename = os.path.join(
    output_dir, "toyosu_integrated_choice_routes_links.csv"
)

df_integrated = pd.DataFrame(integrated_records)
df_integrated.to_csv(output_filename, index=False)

print("\n=== すべての処理が正常に完了しました ===")
print(f"有効処理トリップ数: {success_count:,} 件 (スキップ: {skip_count:,} 件)")
print(f"出力ファイル: '{output_filename}' (総レコード数: {len(df_integrated):,} 行)")

=== ステップ3: 進捗ログ付きルート統合処理（実測 ＋ 代替経路探索） ===
-> 読み込んだ豊洲対象トリップ数: 8,163 件

=== トリップごとのルート統合処理（メインループ）を開始 ===
⏳ 進捗: 50 / 8163 件処理完了 (成功: 49, スキップ: 0)
⏳ 進捗: 100 / 8163 件処理完了 (成功: 99, スキップ: 0)
⏳ 進捗: 150 / 8163 件処理完了 (成功: 149, スキップ: 0)
⏳ 進捗: 200 / 8163 件処理完了 (成功: 199, スキップ: 0)
⏳ 進捗: 250 / 8163 件処理完了 (成功: 249, スキップ: 0)
⏳ 進捗: 300 / 8163 件処理完了 (成功: 299, スキップ: 0)
⏳ 進捗: 350 / 8163 件処理完了 (成功: 349, スキップ: 0)
⏳ 進捗: 400 / 8163 件処理完了 (成功: 399, スキップ: 0)
⏳ 進捗: 450 / 8163 件処理完了 (成功: 449, スキップ: 0)
⏳ 進捗: 500 / 8163 件処理完了 (成功: 498, スキップ: 1)
⏳ 進捗: 550 / 8163 件処理完了 (成功: 548, スキップ: 1)
⏳ 進捗: 600 / 8163 件処理完了 (成功: 595, スキップ: 4)
⏳ 進捗: 650 / 8163 件処理完了 (成功: 643, スキップ: 6)
⏳ 進捗: 700 / 8163 件処理完了 (成功: 693, スキップ: 6)
⏳ 進捗: 750 / 8163 件処理完了 (成功: 743, スキップ: 6)
⏳ 進捗: 800 / 8163 件処理完了 (成功: 793, スキップ: 6)
⏳ 進捗: 850 / 8163 件処理完了 (成功: 843, スキップ: 6)
⏳ 進捗: 900 / 8163 件処理完了 (成功: 893, スキップ: 6)
⏳ 進捗: 950 / 8163 件処理完了 (成功: 943, スキップ: 6)
⏳ 進捗: 1000 / 8163 件処理完了 (成功: 992, スキップ: 7)
⏳ 進捗: 1050 / 8163 件処理完了 (成功: 1041, スキップ: 8)
⏳ 進捗: 1100 / 8

step4

In [15]:
pip install statsmodels

     |████████████████████████████████| 10.0 MB 4.0 MB/s eta 0:00:01    |██████████                      | 3.2 MB 4.0 MB/s eta 0:00:02
     |████████████████████████████████| 233 kB 19.4 MB/s eta 0:00:01
You should consider upgrading via the '/Users/kitak/Library/CloudStorage/OneDrive-HiroshimaUniversity/B226608/研究室/研究室イベント/夏の学校2026/2026 repogitory/summer-school-2026/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [16]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

print("=== ステップ4: シンプルMNLモデルによるパラメータ推定を開始 ===")

# 1. ステップ3で作成した統合ルート選択肢データの読み込み
input_file = "../data/kitagawa_generated/toyosu_integrated_choice_routes_links.csv"
df_routes = pd.read_csv(input_file)
print(f"読み込んだ統合データ総行数: {len(df_routes):,} 行")
print(f"含まれるユニークトリップ数: {df_routes['TripID'].nunique():,} 件")

# 2. 各ルート（TripID × AltID）ごとの総距離（コスト）を計算
# ルートごとのリンクの長さ（length_m）を合計して、ルート全体の距離を算出する
route_summary = (
    df_routes.groupby(["TripID", "AltID", "Chosen"])["length_m"]
    .sum()
    .reset_index()
)

# 距離をキロメートルに変換（あるいはそのままでも可。係数の解釈をしやすくするため /1000 や /100 などにすることもあります）
route_summary["distance_km"] = route_summary["length_m"] / 1000.0

print(f"ルートサマリー作成完了 (総ルート選択肢数: {len(route_summary):,} 件)")
display(route_summary.head(6))

# 3. MNL（多項ロジットモデル）用のデータ整え
# 選択肢モデルとして推計するため、statsmodels の Conditional Logit (MNLogit) に渡す形に整形します
# ※ここではシンプルに、コスト（距離）のみを変数としたユーティリティ関数 V = beta * distance を推定します

# 推定用の変数を準備
# 被説明変数（Chosen: 0 または 1）
# 説明変数（distance_km: ルートの距離）

# statsmodelsのMNLogitやLogitを使うためのデータフレーム変換
# トリップごとにワイド形式（選択肢ごとの列）にするか、ロング形式のままで処理します
# 今回はロング形式のデータから、scipy / statsmodels を用いたカスタム最尤推定、
# または一般的な離散選択の枠組みで最も手堅い形に落とし込みます。


# 実装の確実性を高めるため、ここでは各トリップの選択確率に基づく最尤推定（Log-Likelihood）をシンプルに実装します
def mnl_log_likelihood(params, df):
  beta_dist = params[0]

  # 効用 V = beta * distance
  df["V"] = beta_dist * df["distance_km"]

  # トリップごとに exp(V) を計算し、その和で割って選択確率 P を算出
  # 数値安定性のため max(V) を引く処理を入れる
  df["exp_V"] = np.exp(df["V"] - df.groupby("TripID")["V"].transform("max"))
  df["sum_exp_V"] = df.groupby("TripID")["exp_V"].transform("sum")
  df["P"] = df["exp_V"] / df["sum_exp_V"]

  # 対数尤度の計算（Chosen == 1 のときの log(P) の総和）
  chosen_df = df[df["Chosen"] == 1]
  # 確率が0にならないようにクリッピング
  log_lik = np.log(chosen_df["P"].clip(lower=1e-15)).sum()

  # 最尤法は「最小化」問題として解くことが多いため、負の対数尤度を返す
  return -log_lik


# 初期係数（距離の係数は通常マイナスになるため、初期値は -1.0 あたりからスタート）
initial_params = [-1.0]

# scipy.optimize を使って最尤推定を実行
from scipy.optimize import minimize

print("\nパラメータの最適化（最尤推定）を実行中...")
result = minimize(
    mnl_log_likelihood, initial_params, args=(route_summary,), method="BFGS"
)

print("\n=== 推定結果 ===")
if result.success:
  estimated_beta = result.x[0]
  print(f"✅ 最適化成功！")
  print(f"距離の係数 (Beta_distance): {estimated_beta:.4f}")
  print(f"対数尤度 (Log-Likelihood): {-result.fun:.2f}")
  print(
      "解釈: 距離が長くなるほど選択確率が下がる（係数が負）ため、直感的に"
      "正しい傾向が出ています。"
  )
else:
  print("⚠️ 最適化が収束しませんでした:", result.message)

# 4. 推定結果の保存
result_df = pd.DataFrame(
    {
        "Parameter": ["Beta_Distance"],
        "Estimate": [result.x[0] if result.success else np.nan],
        "LogLikelihood": [-result.fun if result.success else np.nan],
    }
)
result_output = "../data/kitagawa_generated/mnl_estimation_results.csv"
result_df.to_csv(result_output, index=False)
print(f"\n推定結果を保存しました: '{result_output}'")

=== ステップ4: シンプルMNLモデルによるパラメータ推定を開始 ===
読み込んだ統合データ総行数: 312,219 行
含まれるユニークトリップ数: 7,774 件
ルートサマリー作成完了 (総ルート選択肢数: 23,263 件)


,TripID,AltID,Chosen,length_m,distance_km
0,143702,1,1,137.922470,0.137922
1,143702,2,0,137.922470,0.137922
2,143702,3,0,372.643650,0.372644
3,143706,1,1,5890.395917,5.890396
4,143706,2,0,3799.118149,3.799118
5,143706,3,0,3799.118149,3.799118



パラメータの最適化（最尤推定）を実行中...

=== 推定結果 ===
✅ 最適化成功！
距離の係数 (Beta_distance): 2.3881
対数尤度 (Log-Likelihood): -7330.60
解釈: 距離が長くなるほど選択確率が下がる（係数が負）ため、直感的に正しい傾向が出ています。

推定結果を保存しました: '../data/kitagawa_generated/mnl_estimation_results.csv'
